# Fetching Financial Data for Machine and Deep Learning Forecasting

This notebook demonstrates how to fetch and prepare fundamental financial data identical in format to `financial_data.csv` used in **CFA Level III Practical Skills Module: Financial Forecasting Using Machine and Deep Learning**.

### Dataset Background
The original course dataset contains **202 columns** for 8 major companies (`AAPL`, `UNH`, `NVDA`, `XOM`, `GE`, `COST`, `JCI`, `ICE`) spanning quarterly reports from 2001 to 2022:
1. **Company Metadata**: `Company Name`, `Sector`, `Industry`, `Ticker`, `Currency`, `Report Date`, `Fiscal Year`, `Fiscal Period`, `Publish Date`, `Restated Date`.
2. **Standardized Statements**: Income Statement, Balance Sheet, Cash Flow items (e.g. `Revenue`, `Cost of Revenue`, `Gross Profit`, `Operating Income (Loss)`, `Net Income`, `Free Cash Flow`, etc.).
3. **Financial Ratios & Metrics**: `Piotroski F-Score`, `Net Debt / EBITDA`, `Net Debt / EBIT`, `Return on Equity`, `Return on Assets`, `Current Ratio`, `Earnings Per Share, Diluted`, etc.
4. **Engineered Multi-Quarter Lag Features**: Percentage changes from last 1, 2, 3, and 4 quarters for 28 metrics (e.g. `Revenue change from last quarter`, `Revenue change from last  2 quarters`, etc.).
5. **Target Outputs**: `% Change in Quarterly EPS (Target Output)` and `% Change in Quarterly Revenue (Target Output)`.

We provide two complete methods:
- **Method A (`simfin`)**: Direct match to the CFA course dataset schema.
- **Method B (`yfinance`)**: Free, open-source alternative requiring no API key.
- **Feature Engineering Pipeline**: Reconstructs the 1-to-4 quarter lag percentage changes and target forecast variables.

In [ ]:
# Install required packages
!pip install pandas numpy simfin yfinance

## Method A: Fetching Fundamental Data via SimFin (Exact Match)

**SimFin** is the exact source of the standardized fundamental accounting statements and derived ratios (`Publish Date`, `Restated Date`, `Piotroski F-Score`, `Net Debt / EBITDA`) in the course.

1. Register for a free API key at [simfin.com](https://www.simfin.com/)
2. Replace `'free'` with your API key if you have one.

In [ ]:
import os
import pandas as pd
import numpy as np
import simfin as sf
from simfin.names import *

# SimFin setup (use 'free' for sample data, or your registered SimFin API key)
SIMFIN_API_KEY = 'free'  # e.g., 'YOUR_SIMFIN_KEY'
sf.set_api_key(SIMFIN_API_KEY)
sf.set_data_dir('simfin_data/')

# Tickers analyzed in the CFA course module
tickers = ['AAPL', 'UNH', 'NVDA', 'XOM', 'GE', 'COST', 'JCI', 'ICE']
print(f"Target Tickers: {tickers}")

In [ ]:
# Fetch quarterly financial statements and derived ratios
print("Downloading / loading quarterly financial data from SimFin...")
try:
    df_income = sf.load_income(variant='quarterly', market='us')
    df_balance = sf.load_balance(variant='quarterly', market='us')
    df_cashflow = sf.load_cashflow(variant='quarterly', market='us')
    df_derived = sf.load_derived(variant='quarterly', market='us')

    # Filter for target tickers
    df_income = df_income.loc[df_income.index.get_level_values('Ticker').isin(tickers)]
    df_balance = df_balance.loc[df_balance.index.get_level_values('Ticker').isin(tickers)]
    df_cashflow = df_cashflow.loc[df_cashflow.index.get_level_values('Ticker').isin(tickers)]
    df_derived = df_derived.loc[df_derived.index.get_level_values('Ticker').isin(tickers)]

    # Combine all statement tables horizontally
    simfin_combined = pd.concat([df_income, df_balance, df_cashflow, df_derived], axis=1)
    simfin_combined = simfin_combined.loc[:, ~simfin_combined.columns.duplicated()]
    simfin_df = simfin_combined.reset_index()

    print(f"SimFin dataset shape: {simfin_df.shape}")
    display(simfin_df.head())
except Exception as e:
    print(f"SimFin fetch notice: {e}")
    simfin_df = pd.DataFrame()

## Method B: Fetching Quarterly Financial Data via yfinance (No API Key Required)

If you do not have a SimFin API key or want live fundamental statements, `yfinance` allows direct extraction of quarterly statements for any listed ticker free of charge.

In [ ]:
import yfinance as yf
import pandas as pd

def fetch_quarterly_financials_yf(ticker_symbol):
    """
    Fetches quarterly income statement, balance sheet, and cash flow for a ticker via yfinance.
    """
    ticker = yf.Ticker(ticker_symbol)
    
    # Quarterly statements return with dates as columns; transpose so dates are rows
    inc = ticker.quarterly_income_stmt.T if ticker.quarterly_income_stmt is not None else pd.DataFrame()
    bs = ticker.quarterly_balance_sheet.T if ticker.quarterly_balance_sheet is not None else pd.DataFrame()
    cf = ticker.quarterly_cashflow.T if ticker.quarterly_cashflow is not None else pd.DataFrame()
    
    # Join all statements along the Report Date index
    combined = inc.join(bs, lsuffix='_inc', rsuffix='_bs', how='outer')
    combined = combined.join(cf, rsuffix='_cf', how='outer')
    
    combined['Ticker'] = ticker_symbol
    combined.index.name = 'Report Date'
    return combined.reset_index()

# Fetch for the tickers
tickers_to_fetch = ['AAPL', 'GE', 'NVDA', 'UNH', 'XOM', 'COST', 'JCI', 'ICE']
all_yf_dfs = []

for ticker in tickers_to_fetch:
    print(f"Fetching quarterly data for {ticker}...")
    try:
        t_df = fetch_quarterly_financials_yf(ticker)
        all_yf_dfs.append(t_df)
    except Exception as e:
        print(f"Could not fetch {ticker}: {e}")

yf_financials_df = pd.concat(all_yf_dfs, ignore_index=True) if all_yf_dfs else pd.DataFrame()
print(f"yfinance dataset shape: {yf_financials_df.shape}")
display(yf_financials_df.head())

## Feature Engineering: Computing 1-to-4 Quarter Lags & Target Outputs

The notebook `Financial Forecasting Using Machine and Deep Learning.ipynb` uses multi-quarter percentage changes:
- `[metric] change from last quarter`
- `[metric] change from last  2 quarters`
- `[metric] change from last  3 quarters`
- `[metric] change from last  4 quarters`
- Target variables: `% Change in Quarterly EPS (Target Output)` and `% Change in Quarterly Revenue (Target Output)`.

Here is the function that builds these exact features:

In [ ]:
def compute_lags_and_targets(df, ticker_col='Ticker', date_col='Report Date', eps_col=None, rev_col=None):
    """
    Sorts chronologically and calculates 1, 2, 3, 4-quarter lag percentage changes
    and target forecast variables for each company.
    """
    df = df.copy()
    df[date_col] = pd.to_datetime(df[date_col])
    df = df.sort_values(by=[ticker_col, date_col]).reset_index(drop=True)
    
    # Key financial metrics from CFA course
    cfa_metrics = [
        'Shares (Basic)', 'Shares (Diluted)', 'Revenue', 'Operating Income (Loss)', 
        'Net Income', 'Total Assets', 'Total Liabilities & Equity', 'EBITDA', 
        'Free Cash Flow', 'Gross Profit Margin', 'Operating Margin', 'Net Profit Margin', 
        'Return on Equity', 'Return on Assets', 'Free Cash Flow to Net Income', 
        'Current Ratio', 'Liabilities to Equity Ratio', 'Earnings Per Share, Basic', 
        'Earnings Per Share, Diluted', 'Sales Per Share', 'Equity Per Share', 
        'Free Cash Flow Per Share', 'Piotroski F-Score', 'Return On Invested Capital', 
        'Cash Return On Invested Capital', 'Net Debt / EBITDA', 'Net Debt / EBIT'
    ]
    
    # If columns match CFA exactly, use them; otherwise lag available numeric columns
    available_metrics = [m for m in cfa_metrics if m in df.columns]
    if not available_metrics:
        non_metric = [ticker_col, date_col, 'Company Name', 'Sector', 'Industry', 'Currency']
        available_metrics = [c for c in df.columns if c not in non_metric and pd.api.types.is_numeric_dtype(df[c])][:25]
        
    print(f"Calculating multi-quarter lags for {len(available_metrics)} metrics...")
    
    for m in available_metrics:
        for lag in [1, 2, 3, 4]:
            suffix = 'change from last quarter' if lag == 1 else f'change from last  {lag} quarters'
            col_name = f'{m} {suffix}'
            df[col_name] = df.groupby(ticker_col)[m].pct_change(periods=lag)
            
    # Detect or use specified EPS / Revenue columns for target output calculation
    eps_candidate = eps_col or next((c for c in ['Earnings Per Share, Diluted', 'Diluted EPS', 'Basic EPS'] if c in df.columns), None)
    rev_candidate = rev_col or next((c for c in ['Revenue', 'Total Revenue', 'Operating Revenue'] if c in df.columns), None)
    
    if eps_candidate:
        df['% Change in Quarterly EPS (Target Output)'] = df.groupby(ticker_col)[eps_candidate].pct_change().shift(-1)
        print(f"Target created: '% Change in Quarterly EPS (Target Output)' from '{eps_candidate}'")
        
    if rev_candidate:
        df['% Change in Quarterly Revenue (Target Output)'] = df.groupby(ticker_col)[rev_candidate].pct_change().shift(-1)
        print(f"Target created: '% Change in Quarterly Revenue (Target Output)' from '{rev_candidate}'")
        
    return df

In [ ]:
# Apply feature engineering to the fetched dataset
source_df = simfin_df if ('simfin_df' in locals() and not simfin_df.empty) else yf_financials_df

if not source_df.empty:
    processed_df = compute_lags_and_targets(source_df)
    print(f"Processed DataFrame shape: {processed_df.shape}")
    
    # Save as financial_data.csv in the local folder
    csv_path = 'financial_data.csv'
    processed_df.to_csv(csv_path, index=False)
    print(f"Saved dataset to '{csv_path}' successfully!")
else:
    print("No source data found. Run Method A or Method B above.")

## Verification: Reading `financial_data.csv`

Test loading `financial_data.csv` just like cell 8 in `Financial Forecasting Using Machine and Deep Learning.ipynb`.

In [ ]:
# Load and inspect financial_data.csv
df = pd.read_csv('financial_data.csv')

print("DataFrame preview:")
display(df.head())

print("\nSummary info:")
print(f"Number of rows: {df.shape[0]}")
print(f"Number of columns: {df.shape[1]}")
print(f"Available Tickers: {df['Ticker'].unique()}")